# 5 The notebook as handed out, and a study of your own

Notebook 04 ran one search and read it for you. Here you drive, in two parts.

**Part 1 is the notebook as handed out.** A baseline with its seed spread, the encoder
comparison, one search over a given space, its winner rerun over seeds, and a noise
ablation. You change nothing in it; every student's Part 1 is the same, which is what
makes it a check: if your baseline differs from the class's, something in your setup
differs, and that is worth finding before anything else.

**Part 2 is your own study.** You have an idea you want to test, design the search space
that tests it, give the study a name, and run it. Every run of the study carries its name
and its search space, so what reaches the next lesson is the runs themselves: two sets per
student, the handed-out ones and your study. A study that disappointed is a result worth
bringing. "I expected X, measured Y, and here is the seed spread that makes me believe it"
is the habit this lesson exists for, whichever way it came out.

Everything starts on **DTD**: 47 texture classes, roughly twenty points of headroom, and
an encoder axis that actually moves.

In [ ]:
import os
import warnings

os.environ.setdefault("RAY_ENABLE_UV_RUN_RUNTIME_ENV", "0")     # see notebook 04
warnings.filterwarnings("ignore", message=".*pkg_resources.*")

import sys
import time
from functools import cache
from pathlib import Path

import datasets
import matplotlib.pyplot as plt
import numpy as np
import torch
from loguru import logger
from torch import nn
from torch.utils.data import DataLoader
from vectormesh import VectorCache
from vectormesh.components import Gate, GaussianNoise, NeuralNet, Projection, Serial, Skip
from vectormesh.components.metrics import Accuracy
from vectormesh.data import Collate, OneHot
from vectormesh.training import Trainer, TrainerSettings

from notebooktester import param
from runlog import Study, log_run, space_tags, use_experiment

logger.remove()
logger.add(sys.stderr, level="WARNING")
datasets.disable_progress_bars()

EPOCHS = param(15, test=1)
N_TRIALS = param(16, test=2)
SEEDS = param(3, test=2)

DATASET = "dtd"
ENCODERS = ["dinov2-small", "resnet-18", "mobilenet_v2_1.0_224"]

# Every published cache, by dataset and encoder. A study on another dataset or encoder
# picks its cache from here.
CACHES = {
    "dtd": {enc: f"pttrn-io/dtd-{enc}" for enc in
            ["dinov2-small", "dinov2-base", "dinov2-large", "resnet-18", "mobilenet_v2_1.0_224"]},
    "eurosat": {enc: f"pttrn-io/eurosat-{enc}" for enc in ENCODERS},
    "oxford-flowers": {enc: f"pttrn-io/oxford-flowers-{enc}" for enc in ENCODERS},
    "fashion-mnist": {enc: f"pttrn-io/fashion-mnist-{enc}" for enc in
                      ["dinov2-small", "mobilenet_v2_1.0_224"]},
}

# Part 1 logs to `lesson1-notebook`, through runlog.log_run, in the shape the class run
# protocol expects; Part 2 logs to `lesson1-study` through a Study. You submit both at the
# end of the notebook, and the next lesson opens with the whole class's runs side by side.
use_experiment("lesson1-notebook")

In [ ]:
def load(dataset: str, encoder: str, split: str) -> VectorCache:
    if encoder not in CACHES.get(dataset, {}):
        raise KeyError(f"no published cache for {dataset!r} with {encoder!r}; "
                       f"CACHES has {({d: list(e) for d, e in CACHES.items()})}")
    return VectorCache.from_hub(CACHES[dataset][encoder], split=split)


@cache
def shape_of(dataset: str, encoder: str) -> tuple[int, int]:
    """(vector width, number of classes) of a cache."""
    train = load(dataset, encoder, "train")
    return train.dim, train.num_classes


def loaders(dataset: str, encoder: str, batch_size: int = 64, n_train: int | None = None):
    # n_train shrinks the training set for the data-size questions below.
    # The shuffle is NOT optional -- see the cell after this one.
    out = []
    for split in ("train", "test"):
        cache = load(dataset, encoder, split)
        ds = cache.dataset
        if n_train is not None and split == "train":
            ds = ds.shuffle(seed=0).select(range(n_train))
        oh = OneHot(num_classes=cache.num_classes, label_col="label", target_col="onehot")
        col = Collate(embedding_col="embed", target_col="onehot", padder=torch.stack)
        out.append(DataLoader(ds.map(oh), batch_size=batch_size,
                              shuffle=(split == "train"), collate_fn=col))
    return out


class TimedTrainer(Trainer):
    """A Trainer that adds up the seconds spent in training steps. Evaluation after every
    epoch is not counted: `train_seconds` is the cost of fitting, not of scoring."""

    train_seconds = 0.0

    def trainbatches(self) -> float:
        start = time.perf_counter()
        loss = super().trainbatches()
        self.train_seconds += time.perf_counter() - start
        return loss


def run(model, train_dl, test_dl, lr: float = 1e-3, epochs: int = EPOCHS) -> tuple[float, float]:
    """Train `model` and return (test accuracy, seconds spent in training steps)."""
    # vectormesh.training.Trainer, same as notebook 04: log_dir is computed but never
    # created on disk unless something writes to it, and progress=False drops the tqdm
    # bars, which is what makes calling this once per seed -- several times per exercise
    # below -- cheap. `.eval()`/`.train()`/`no_grad()` are handled internally too.
    settings = TrainerSettings(
        epochs=epochs,
        metrics=[Accuracy()],
        logdir=Path("modellogs").resolve(),
        train_steps=len(train_dl),
        valid_steps=len(test_dl),
        optimizer_kwargs={"lr": lr},
        earlystop_kwargs=None,
    )
    trainer = TimedTrainer(
        model=model,
        settings=settings,
        loss_fn=nn.CrossEntropyLoss(),
        optimizer=torch.optim.Adam,
        traindataloader=train_dl,
        validdataloader=test_dl,
        progress=False,
    )
    return trainer.loop().metric_dict["Accuracy"], trainer.train_seconds


# Every choice a head can make, with the value that leaves it out. A run logs all of them
# (see run_params), so each is an axis in the class's parallel-coordinates plot.
HEAD = {"n_hidden_layers": 0, "hidden_dim": 0, "sigma": 0.0, "norm": "none",
        "skip": False, "gate": False, "lr": 1e-3}


def make_head(config: dict, dim: int, n_classes: int) -> Serial:
    # the same builder as notebook 04 §4.1: 0 hidden layers is the baseline's single Linear,
    # more layers go through one NeuralNet call, which puts a GELU between every Linear.
    # The switches norm ("batch", "layer" or "none"), skip and gate change the hidden block;
    # they do nothing without hidden layers. Add an axis to your param_space? Use it here,
    # and give it a default in HEAD.
    c = {**HEAD, **config}
    n_hidden, width = c["n_hidden_layers"], c["hidden_dim"]
    if n_hidden == 0:
        head = nn.Linear(dim, n_classes)
    elif c["norm"] == "none" and not c["skip"] and not c["gate"]:
        head = NeuralNet([dim, *[width] * n_hidden], n_classes)
    else:
        layers, d = [], dim
        for _ in range(n_hidden):
            layers.append(nn.Linear(d, width))
            if c["norm"] == "batch":
                layers.append(nn.BatchNorm1d(width))
            elif c["norm"] == "layer":
                layers.append(nn.LayerNorm(width))
            layers.append(nn.GELU())
            d = width
        block = Serial(layers)
        if c["skip"]:   # transform(norm(x)) + projection(norm(x)), see vectormesh's Skip
            block = Skip(block, in_size=dim, projection=Projection(dim, width))
        head = Serial([block, *([Gate(width)] if c["gate"] else []), nn.Linear(width, n_classes)])
    return Serial([GaussianNoise(sigma=c["sigma"]), head])


def run_params(dataset: str, encoder: str, config: dict | None = None, **extra) -> dict:
    """The params of a run: the vectors it starts from, every head choice, the budget."""
    return {"dataset": dataset, "encoder": encoder, "embed_dim": shape_of(dataset, encoder)[0],
            **HEAD, **{k: v for k, v in (config or {}).items() if k not in ("dataset", "encoder")},
            "epochs": EPOCHS, **extra}


def seed_spread(make_model, train_dl, test_dl, seeds: int = SEEDS, **kwargs) -> tuple[np.ndarray, float]:
    """Accuracy per seed, and the mean seconds spent in training steps."""
    scores, seconds = [], []
    for seed in range(seeds):
        torch.manual_seed(seed)
        accuracy, spent = run(make_model(), train_dl, test_dl, **kwargs)
        scores.append(accuracy)
        seconds.append(spent)
    return np.array(scores), float(np.mean(seconds))


def compare(name: str, scores: np.ndarray, versus: np.ndarray | None = None) -> None:
    line = f"{name:<32} {scores.mean():.4f} +/- {scores.std():.4f}   (n={len(scores)})"
    if versus is not None:
        diff = scores.mean() - versus.mean()
        pooled = np.sqrt(scores.var(ddof=1) / len(scores) + versus.var(ddof=1) / len(versus))
        line += f"   {100 * diff:+.2f} pt, {diff / pooled:+.1f} sigma"
    print(line)


DIM = {enc: shape_of(DATASET, enc)[0] for enc in ENCODERS}
N_CLASSES = shape_of(DATASET, "dinov2-small")[1]
print("widths:", DIM, "| classes:", N_CLASSES)

# Part 1 — the notebook as handed out

### First, look at the data you are about to slice

That `.shuffle(seed=0)` above is not defensive habit, it is load-bearing, and the reason is
worth two minutes because it is the kind of bug that produces a *plausible* wrong answer
rather than a crash.

In [ ]:
dtd = load(DATASET, "dinov2-small", "train")
labels = dtd["label"][:]
shuffled_labels = dtd.dataset.shuffle(seed=0)["label"][:]

print("first 20 labels in the DTD cache:", labels[:20].tolist())
print()
print(f"{'rows taken':>12} {'classes, as stored':>20} {'classes, shuffled':>19}")
for n in (470, 940, 1880):
    print(f"{n:>12} {len(set(labels[:n].tolist())):>14}/{N_CLASSES:<5} "
          f"{len(set(shuffled_labels[:n].tolist())):>13}/{N_CLASSES}")

**The DTD cache is stored grouped by label.** Take the first 940 rows and you do not get a
smaller sample of the task — you get a *different, easier* task with a quarter of the
classes, and a test set full of categories your model has never once seen.

The reason this is dangerous rather than merely wrong is that it fails *quietly and
plausibly*. Subsampling this way at 470 / 940 / 1880 rows produces test accuracies of
0.12 / 0.24 / 0.44, which looks exactly like a healthy learning curve — right up until you
notice those numbers are just `n / 3760`. The model was scoring near-perfectly on the
classes it had and zero on the rest. Shuffled, the same sizes give about 0.69 / 0.75 / 0.78
(measured with a one-hidden-layer head; `planning/evidence.md` §3b), which
is the real curve and a far less exciting one.

Two general lessons, both cheap:

- **Not every cache is shuffled.** The EuroSAT caches are (they were subsampled at build
  time, which shuffles); the DTD ones were built `--full`, which does not. Do not assume —
  it is one line to check, as above.
- **A number that looks like a textbook result deserves more suspicion, not less.** The
  broken version of this exercise produced a *nicer* curve than the correct one.

## E1 — The baseline, first

**Before you search anything**, establish the two-line baseline on DTD and its seed
spread. A search without a baseline is unreadable; a baseline without a spread cannot tell
you whether a later "win" is real.

Every number you produce below gets compared against these two.

In [ ]:
train_dl, test_dl = loaders(DATASET, "dinov2-small")

baseline, baseline_seconds = seed_spread(lambda: nn.Linear(DIM["dinov2-small"], N_CLASSES),
                                         train_dl, test_dl)
compare("baseline: Linear(384, 47)", baseline)
print(f"\n=> a later result must clear {baseline.mean() + 2 * baseline.std():.4f} "
      f"before you should believe it beat this.")

example = next(iter(train_dl))[0][:2]   # two rows are enough to trace every layer's shape
log_run("baseline", "baseline", dataset=DATASET, scores={"accuracy": baseline},
        params=run_params(DATASET, "dinov2-small"), train_seconds=baseline_seconds,
        model=nn.Linear(DIM["dinov2-small"], N_CLASSES), example=example)

Both numbers are now in mlflow, logged before anything else ran. That order is the point:
a baseline fixed in advance is the only thing that stops you from quietly redefining what
counted as success once you have seen the results.

## E2 — Argue from the metadata, then check

Notebook 01 said the metadata is the public API of a cache. This is not a cold guess —
you already have evidence, so use it.

**Before running the next cell**, answer for yourself:

1. Look back at notebook 01 §1.5's table: DINOv2 beat resnet-18 and mobilenet by 9-15
   points on *flowers*. Textures are a different domain, but the *ranking* of general-purpose
   image encoders tends to be fairly stable across natural-image tasks. Would you expect the
   same ordering here, or is there a reason textures specifically might reshuffle it? Either
   answer is defensible — the point is to have reasoned about it, not to have guessed right.
2. `mobilenet_v2` produces 1280 numbers per image against DINOv2's 384. Notebook 01 §1.5
   already told you width is not quality on flowers (`mobilenet_v2` lost by eleven points
   there despite more dimensions). Do you expect that to hold here too?
3. On flowers the encoder spread was about 15 points and on EuroSAT about 3. DTD's spread is
   somewhere in that range — pick a number and say why textures should be closer to one end
   than the other.

Then run it.

In [ ]:
results = {}
for enc in ENCODERS:
    tr, te = loaders(DATASET, enc)
    torch.manual_seed(0)
    results[enc], seconds = run(nn.Linear(DIM[enc], N_CLASSES), tr, te)
    log_run(f"encoder-{enc}", "ablation", dataset=DATASET, scores={"accuracy": [results[enc]]},
            params=run_params(DATASET, enc), train_seconds=seconds,
            model=nn.Linear(DIM[enc], N_CLASSES), example=torch.zeros(2, DIM[enc]))

print(f"{'encoder':<24} {'width':>6} {'accuracy':>9}")
print("-" * 42)
for enc, acc in sorted(results.items(), key=lambda kv: -kv[1]):
    print(f"{enc:<24} {DIM[enc]:>6} {acc:>9.4f}")

spread = max(results.values()) - min(results.values())
print(f"\nspread across encoders: {100 * spread:.1f} pt")
print(f"for comparison, the baseline's own seed spread: {100 * baseline.std():.2f} pt")

Compare that last pair of numbers. **How many seed-spreads wide is the encoder axis?** That
ratio is the honest way to say "this axis matters and that one does not".

## E3 — The handed-out search

The search below is the one handed out to the whole class: `dinov2-small` only, a linear
head or one hidden layer, some input noise, a learning-rate range. It is someone's design,
not the only one; in Part 2 you design your own. Here, run it as it is, so that your search
can be compared with everyone else's.

`trainable` and `search` are the same machinery Part 2 uses: `trainable` reads the dataset
and the encoder from the config, and `search` runs `N_TRIALS` trials of a space and logs
every trial through the `log` function it is given.

In [ ]:
from ray import tune
from ray.tune.search.hyperopt import HyperOptSearch


def trainable(config):
    tr, te = loaders(config["dataset"], config["encoder"])
    train = load(config["dataset"], config["encoder"], "train")
    model = make_head(config, train.dim, train.num_classes)

    # report every epoch, not only the last, so each trial's learning curve reaches mlflow
    def to_ray(epoch, train_loss, test_loss, metric_dict):
        tune.report({"Accuracy": metric_dict["Accuracy"], "train_loss": train_loss,
                     "test_loss": test_loss, "train_seconds": trainer.train_seconds})

    torch.manual_seed(0)
    settings = TrainerSettings(
        epochs=config["epochs"],
        metrics=[Accuracy()],
        logdir=Path("ray_results/trials").resolve(),  # never created unless earlystop saves
        train_steps=len(tr),
        valid_steps=len(te),
        optimizer_kwargs={"lr": config["lr"]},
        earlystop_kwargs=None,
    )
    trainer = TimedTrainer(
        model=model,
        settings=settings,
        loss_fn=nn.CrossEntropyLoss(),
        optimizer=torch.optim.Adam,
        traindataloader=tr,
        validdataloader=te,
        progress=False,  # ray's own status table already shows progress across trials
        reporters=[to_ray],
    )
    trainer.loop()


def search(space: dict, name: str, log) -> tuple:
    """Run N_TRIALS trials of `space`, log every trial through `log`, and return the
    results with a table of them (one row per trial).

    The search algorithm is seeded, so the same space draws the same trials on every
    laptop: everyone's handed-out search is the same search."""
    tuner = tune.Tuner(
        trainable,
        param_space=space,
        tune_config=tune.TuneConfig(metric="Accuracy", mode="max",
                                    search_alg=HyperOptSearch(random_state_seed=0),
                                    num_samples=N_TRIALS),
        run_config=tune.RunConfig(
            storage_path=Path("ray_results").resolve(),
            name=name,
            # verbose=1, no explicit reporter: ray auto-detects a running Jupyter kernel and
            # renders the same live status table 4_tuning_networks/01_hypertuner.ipynb gets.
            verbose=1,
        ),
    )
    start = time.time()
    found = tuner.fit()

    # Every trial becomes its own mlflow run, with its learning curve and its architecture.
    # They are logged here, after the search, and not from inside the trials: ray runs each
    # trial in its own process, and the sqlite store wants one writer at a time.
    for i, result in enumerate(found):
        if result.error or result.metrics_dataframe is None:
            continue
        config = dict(result.config)
        curves = result.metrics_dataframe[["Accuracy", "train_loss", "test_loss"]]
        dim, n_classes = shape_of(config["dataset"], config["encoder"])
        log(f"trial-{i:02d}", "trial", dataset=config["dataset"],
            params=run_params(config["dataset"], config["encoder"], config),
            train_seconds=float(result.metrics_dataframe["train_seconds"].iloc[-1]),
            curves={"accuracy": curves["Accuracy"].tolist(),
                    "train_loss": curves["train_loss"].tolist(),
                    "test_loss": curves["test_loss"].tolist()},
            model=make_head(config, dim, n_classes), example=torch.zeros(2, dim),
            tags=space_tags(space))

    table = found.get_dataframe()
    table = table[["Accuracy"] + [c for c in table.columns if c.startswith("config/")
                                  and c not in ("config/epochs", "config/dataset")]]
    table.columns = [c.replace("config/", "") for c in table.columns]
    print(f"{N_TRIALS} trials in {time.time() - start:.0f}s, each logged to mlflow\n")
    print(table.sort_values("Accuracy", ascending=False).head(6).to_string(index=False))
    return found, table


# The search space handed out to the whole class. Part 2 starts from a copy of it.
HANDED_OUT_SPACE = {
    "dataset": DATASET,
    "encoder": tune.choice(["dinov2-small"]),
    "n_hidden_layers": tune.choice([0, 1]),
    "hidden_dim": tune.choice([64, 256]),   # ignored by trainable when n_hidden_layers=0
    "sigma": tune.uniform(0.0, 0.5),
    "lr": tune.loguniform(3e-4, 3e-3),
    "epochs": EPOCHS,
}

In [ ]:
handed_out, df = search(HANDED_OUT_SPACE, "handed_out_search", log_run)
print(f"\nbest trial : {df['Accuracy'].max():.4f}")
print(f"baseline   : {baseline.mean():.4f} +/- {baseline.std():.4f}")

In [ ]:
from plotting import search_parallel_plot

# Same plot as notebook 04 §4.2, generalised to whatever columns a param_space produces --
# one line per trial, every axis at once, so an interaction between two axes is still
# visible even though no single-axis panel would show it.
fig = search_parallel_plot(df, columns=list(df.columns), color="Accuracy")
fig.update_layout(title="The handed-out search, every trial and every axis at once")
fig.show()

**Did anything beat the baseline? By how much?**

Careful with the comparison you just printed. The best trial is a **single seed**, selected
as the maximum of 16 draws. The baseline is a **mean over several seeds**. Taking the max
of many noisy numbers and comparing it to an average is one of the most common ways to
manufacture a win that is not there — the max of 16 draws from the baseline's own
distribution would already sit above the baseline's mean.

Which is exactly what E4 is for.

## E4 — Rerun the winner honestly

Take the winning configuration and rerun it across seeds. Then compare like with like.

In [ ]:
best = handed_out.get_best_result().config
print("winning config:", {k: v for k, v in best.items() if k != "epochs"})
best_dim, _ = shape_of(best["dataset"], best["encoder"])


def make_winner():
    return make_head(best, best_dim, N_CLASSES)


win_tr, win_te = loaders(best["dataset"], best["encoder"])
winner, winner_seconds = seed_spread(make_winner, win_tr, win_te, lr=best["lr"])

print()
compare("baseline", baseline)
compare("search winner", winner, versus=baseline)
print(f"\nsingle-seed best from the search : {df['Accuracy'].max():.4f}")
print(f"the same config over {SEEDS} seeds     : {winner.mean():.4f}")

In [ ]:
# The number that stands for this search: the winner rerun over seeds, not its single-seed
# leaderboard score. In the class's mlflow it sits next to everyone else's.
log_run("winner", "final", dataset=best["dataset"], scores={"accuracy": winner},
        params=run_params(best["dataset"], best["encoder"], best), train_seconds=winner_seconds,
        model=make_winner(), example=torch.zeros(2, best_dim), tags=space_tags(HANDED_OUT_SPACE))

**This is the number that stands for the search**, not the leaderboard number.

Answer these for yourself before moving on:

1. How much of the apparent win survived being rerun? If it shrank, by how much?
2. Is the difference larger than the pooled sigma printed above? If not, what is the honest
   one-sentence conclusion?
3. Most "wins" at this scale are inside seed noise. Is this one? Commit to an answer.

## E5 — An ablation you can actually settle

`GaussianNoise` is the one head-side knob with any prior evidence behind it: earlier
measurements had it worth about +0.4 points on DTD. That is small enough that it is a
genuinely open question rather than a known result.

**Predict first:** does input noise still help once the training set is full size? Argue
from what noise is *for* — then test it at two training sizes.

In [ ]:
FULL = 3760


def with_noise(sigma):
    return lambda: Serial([GaussianNoise(sigma=sigma),
                           nn.Linear(DIM["dinov2-small"], N_CLASSES)])


def train_accuracy(model, train_dl) -> float:
    model.eval()
    correct = total = 0
    with torch.no_grad():
        for X, y in train_dl:
            correct += int((model(X).argmax(1) == y.argmax(1)).sum())
            total += len(y)
    return correct / total


for n_train in (param(940, test=200), None):
    tr, te = loaders(DATASET, "dinov2-small", n_train=n_train)
    rows = n_train or FULL
    # scale epochs so every training size gets a comparable number of gradient steps,
    # otherwise you are measuring "less data" and "less training" at the same time
    epochs = max(EPOCHS, round(EPOCHS * FULL / rows))

    print(f"\n--- {rows} rows, {epochs} epochs ---")
    clean, clean_seconds = seed_spread(with_noise(0.0), tr, te, epochs=epochs)
    noisy, noisy_seconds = seed_spread(with_noise(0.3), tr, te, epochs=epochs)
    compare("sigma = 0.0", clean)
    compare("sigma = 0.3", noisy, versus=clean)
    for sigma, scores, seconds in ((0.0, clean, clean_seconds), (0.3, noisy, noisy_seconds)):
        log_run(f"noise-{sigma}-{rows}rows", "ablation", dataset=DATASET,
                scores={"accuracy": scores}, train_seconds=seconds,
                params=run_params(DATASET, "dinov2-small", {"sigma": sigma},
                                  n_train=rows, epochs=epochs),
                model=with_noise(sigma)(), example=torch.zeros(2, DIM["dinov2-small"]))

    torch.manual_seed(0)
    fitted = with_noise(0.0)()
    run(fitted, tr, te, epochs=epochs)
    print(f"{'training-set accuracy':<32} {train_accuracy(fitted, tr):.4f}")

Note the epoch scaling in that cell. Without it you would be changing two things at once —
less data *and* fewer gradient steps — and could not attribute the difference to either.
Was that the right call?

Three things to take from the output, and the third is the one worth keeping.

**Check whether either difference cleared its sigma.** If it did not, the honest sentence is
"no measurable effect at this training size", not "noise helped slightly". Resist the pull
of the sign: a +0.1 pt difference at 0.4 sigma is not a small benefit, it is no evidence.

**"Does GaussianNoise help?" is not a question with an answer.** "Does it help at *this*
training size, on *this* representation?" is. Regularisation is supposed to buy something
exactly when a model can memorise its training set — so look at the last line.

**The head reaches ~100% training accuracy and the test number does not move.** That is the
informative result. The model is memorising completely, and regularising it harder changes
nothing downstream, which means the ceiling you are hitting is not head capacity — it is
what the frozen representation makes linearly separable. You cannot regularise your way to
a better representation. Notebook 04 reached the same conclusion from the other direction:
the encoder moves accuracy by more than ten points, the head knobs by about two.

That is the end of the handed-out part. Everything it logged is in the experiment
`lesson1-notebook`.

# Part 2 — your own study

One study of your own: an idea you want to test, a search space designed to test it, and
the same discipline as Part 1 — a baseline, a search, the winner rerun over seeds, and the
difference in sigmas. Some ideas that are cheap and genuinely open:

- Does the encoder ranking from E2 hold on a *different* dataset (`eurosat`), or is it a
  property of textures specifically?
- Does fusion (notebook 04 §4.3) behave differently when the two encoders are close in
  strength — say `resnet-18` + `mobilenet_v2` — than when one clearly dominates?
- Does the gap between the encoders widen or narrow as you shrink the training set? Which
  direction do you expect, and why?
- One hidden layer beats the linear baseline by about two points on DTD. Does that gain hold
  on EuroSAT, where the linear head already scores around 0.94?
- Width and depth of the head (`hidden_dim`, `n_hidden_layers`) against accuracy, and against
  what they cost: `train_seconds` and `n_params`. Notebook 04 found added depth losing on a
  frozen encoder; where does more width stop paying?
- Normalisation in the hidden block (`norm`: `"batch"`, `"layer"` or `"none"`). The vectors
  come out of a frozen encoder already on a fixed scale; does normalising inside the head
  still buy anything?
- A skip connection around the hidden block (`skip`), vectormesh's `Skip`: the block's
  output plus a projection of its input. With one or two hidden layers there is little depth
  to skip; does it change anything here?
- A gate on the hidden block (`gate`), vectormesh's `Gate`: `sigmoid(W·h) * h`. Gating has so
  far tied its capacity-matched control in this course's measurements; is that true for
  your head too?

**Read as a parallel-coordinates plot.** In the lesson the class's runs are read in mlflow
as one parallel-coordinates plot: the choices on the axes (`encoder`, `embed_dim`,
`n_hidden_layers`, `hidden_dim`, `norm`, `skip`, `gate`, `lr`, ...) and the outcomes
(`accuracy`, `n_params`, `train_seconds`) at the end. Every run logs all of them. A study
that varies one switch with everything else equal is the easiest to read there: its lines
differ on one axis, so whatever the outcome axes do, that axis did it.

**Prepare for the lesson.** The next lesson discusses the class's runs live, in mlflow, and
for your study you will be asked three things:

1. **Why did you do this?** What made this the question worth your budget?
2. **What was the idea behind it?** What did you expect to happen, and from what?
3. **What did you learn when it disappointed?** A low accuracy can be the interesting
   result: a search that found nothing, or a winner that shrank back into the baseline's
   noise once rerun, says something about the representation, the space or the idea.

Think these through for yourself, before you run and after. Nothing is written down or
handed in: the runs are what you bring, and the answers are what you say about them.

**First, name the study.** The name is how the class finds your runs in mlflow: lowercase
letters, digits and dashes, such as `depth-on-eurosat`. The study cell below refuses to run
while `STUDY` still holds its placeholder.

In [ ]:
STUDY = "name-your-study"

**Then design the space**, in the cell below. It starts as a copy of the handed-out one;
every line you keep is a decision too. Decide these before you run anything:

- **Should `encoder` be in the space at all?** You have measured that axis directly in E2.
  Leaving it in means most of your budget gets spent re-confirming what you already know.
  Taking it out means your search can only explore the flat part. Which is the better use
  of 16 trials? There is a defensible answer either way — pick one and know why.
- **What is the widest hidden layer worth trying?** Notebook 04's evidence says added depth
  loses on a frozen encoder. Is your space designed to *find* a good architecture, or to
  *test* that idea? Those want different ranges.
- **How wide a learning-rate range?** Too narrow and you have assumed the answer; too wide
  and you burn trials on configs that cannot train.
- **Which dataset?** `dataset` is an axis like any other; every name in `CACHES` works, with
  any encoder listed for it there.

That is the whole knob-set the handed-out space uses, and it is not the only one. A few
ways to widen the design space, none required, all cheap to try:

- **A `Parallel` fusion branch**, notebook 04 §4.3's construction, as one more entry in the
  space instead of a separate experiment — does giving the search the *option* of fusion
  change what it picks, compared to deciding by hand?
- **More encoders than the ones published here.** Every one you have used so far was
  chosen for you. If you want to try another — a CLIP variant, a newer DINOv3, a
  domain-specific encoder — an LLM with web or an MCP tool for browsing the HuggingFace Hub
  can shortlist candidates faster than reading model cards one by one; a defensible
  rationale still has to be *yours* (what was it pretrained on, and does that resemble
  textures?). Once you have picked one, `01c_build_your_own_cache.ipynb` (optional, in this
  same folder) builds a small cache for it on any hardware; add it to `CACHES`.
- **A different kind of head knob entirely** — dropout, a residual connection, weight decay.
  None of these were in the space you were handed, which was itself a choice, not a law.
  A new axis needs a line in `make_head` that uses it.

Your space travels with every run of the study: one tag per axis (`space_encoder`,
`space_lr`, ...), the tag `searched_axes` listing the axes you searched, the one line
`search_space`, and the whole of it in `search_space.json`. In mlflow the class can filter
on them, for example `tags.searched_axes LIKE '%encoder%'` for every study that searched
the encoder.

In [ ]:
# ---- YOUR SEARCH SPACE. It starts as a copy of the handed-out one; make it test your idea. ----
param_space = {
    "dataset": "dtd",
    "encoder": tune.choice(["dinov2-small"]),
    "n_hidden_layers": tune.choice([0, 1]),
    "hidden_dim": tune.choice([64, 256]),   # ignored by trainable when n_hidden_layers=0
    "sigma": tune.uniform(0.0, 0.5),
    "lr": tune.loguniform(3e-4, 3e-3),
    "epochs": EPOCHS,
    # switches for the hidden block; make_head builds them, HEAD holds their defaults
    # "norm": tune.choice(["none", "batch", "layer"]),
    # "skip": tune.choice([True, False]),
    # "gate": tune.choice([True, False]),
}

In [ ]:
study = Study(STUDY, space=param_space, default_space=HANDED_OUT_SPACE,
              budget={"n_trials": N_TRIALS, "epochs": EPOCHS, "seeds": SEEDS})
print(f"study    : {study.name}")
print(f"space    : {study.tags['search_space']}")
print(f"searched : {study.tags['searched_axes']}")
print(f"same space as handed out: {study.tags['space_is_default']}")

In [ ]:
mine, my_df = search(param_space, f"study_{study.name}", study.log_run)

**Rerun your winner over seeds, against a baseline on the same dataset.** When the winner
is on DTD, the baseline is the one from E1; on another dataset, the cell measures the same
linear head on `dinov2-small` there first. Both are logged through the study, so the class
sees your comparison as you saw it.

In [ ]:
my_best = mine.get_best_result().config
my_dataset = my_best["dataset"]
print("winning config:", {k: v for k, v in my_best.items() if k != "epochs"})

base_dim, base_classes = shape_of(my_dataset, "dinov2-small")
if my_dataset == DATASET:
    my_baseline, my_baseline_seconds = baseline, baseline_seconds
else:
    base_tr, base_te = loaders(my_dataset, "dinov2-small")
    my_baseline, my_baseline_seconds = seed_spread(
        lambda: nn.Linear(base_dim, base_classes), base_tr, base_te)
study.log_run("baseline", "baseline", dataset=my_dataset, scores={"accuracy": my_baseline},
              params=run_params(my_dataset, "dinov2-small"), train_seconds=my_baseline_seconds,
              model=nn.Linear(base_dim, base_classes), example=torch.zeros(2, base_dim))

my_dim, my_classes = shape_of(my_dataset, my_best["encoder"])


def make_my_winner():
    return make_head(my_best, my_dim, my_classes)


my_tr, my_te = loaders(my_dataset, my_best["encoder"])
my_winner, my_winner_seconds = seed_spread(make_my_winner, my_tr, my_te, lr=my_best["lr"])
study.log_run("winner", "final", dataset=my_dataset, train_seconds=my_winner_seconds,
              params=run_params(my_dataset, my_best["encoder"], my_best),
              scores={"accuracy": my_winner}, model=make_my_winner(),
              example=torch.zeros(2, my_dim))

print()
compare(f"baseline on {my_dataset}", my_baseline)
compare("your search winner", my_winner, versus=my_baseline)
print(f"\nsingle-seed best from your search : {my_df['Accuracy'].max():.4f}")

**Read it in sigmas, not in points**: inside about two, the difference is not a result,
whatever its sign. If it disappointed, that is the third question from above, and the runs
reach the class as they are.

## A second arm of your own

A search answers "what is the best configuration in this space". Many ideas are sharper
than that: *this* change against *that* control, everything else equal. The cell below
builds both arms from the helpers in this notebook, runs each over `SEEDS` seeds, and logs
them through your study with the role `hypothesis`.

As handed out it tests one of the ideas above: one hidden layer against the linear head on
EuroSAT. Change `ARM_DATASET`, `ARM_ENCODER`, `control` and `treatment` to the comparison
your own idea needs; `make_head` builds any head the search can, and `loaders` and
`shape_of` find any cache in `CACHES`.

In [ ]:
ARM_DATASET, ARM_ENCODER = "eurosat", "dinov2-small"
control = {"n_hidden_layers": 0, "hidden_dim": 0, "sigma": 0.0, "lr": 1e-3}
treatment = {"n_hidden_layers": 1, "hidden_dim": 256, "sigma": 0.0, "lr": 1e-3}


def arm(name: str, config: dict) -> np.ndarray:
    """Run one arm over SEEDS seeds and log it through the study as a hypothesis run."""
    dim, n_classes = shape_of(ARM_DATASET, ARM_ENCODER)
    tr, te = loaders(ARM_DATASET, ARM_ENCODER)
    scores, seconds = seed_spread(lambda: make_head(config, dim, n_classes), tr, te,
                                  lr=config["lr"])
    study.log_run(f"hypothesis-{name}", "hypothesis", dataset=ARM_DATASET,
                  params=run_params(ARM_DATASET, ARM_ENCODER, config, arm=name),
                  train_seconds=seconds,
                  scores={"accuracy": scores},
                  model=make_head(config, dim, n_classes), example=torch.zeros(2, dim))
    return scores


control_scores = arm("control", control)
treatment_scores = arm("treatment", treatment)
compare(f"control   ({ARM_DATASET})", control_scores)
compare(f"treatment ({ARM_DATASET})", treatment_scores, versus=control_scores)

---

## Submit your runs before the next lesson

Every cell above logged its runs to mlflow: Part 1 in the experiment `lesson1-notebook` (the
baseline, the encoder comparison, each trial of the handed-out search with its learning
curve, the winner rerun over seeds, the noise ablation), and Part 2 in `lesson1-study` (your
trials, your baseline and winner, your two arms), every one of them carrying your study's
name and search space. Each run has its architecture, with the tensor shape at every layer,
in its `architecture.json`.

Send them to the class runs repository. From the root of the course repository, first look
at what would be submitted:

```bash
uv run scripts/submit_runs.py --lesson 1 --list
```

Both experiments should be in that list. Then submit:

```bash
uv run scripts/submit_runs.py --lesson 1
```

Or tell your coding agent: *"follow docs/submit-runs.md for lesson 1"*. That file walks it
through the one-time setup (cloning the runs repository next to this one) and the
submission. Submitting again later is safe; only new runs are added.

### Reading the runs as a parallel-coordinates plot

In `mlflow ui`, open an experiment, switch to the **Chart** view, then **Add chart →
Parallel coordinates**. Put the params on the axes: `encoder` and `embed_dim` (the vectors a
run starts from), `n_hidden_layers`, `hidden_dim`, `sigma`, `norm`, `skip`, `gate`, `lr`;
and the metrics `accuracy`, `n_params` and `train_seconds` as the outcomes. Filter first, in
the search box, on one dataset and one study (`tags.dataset = 'dtd' and tags.study =
'your-study'`): lines of different datasets do not belong on one axis.

The next lesson opens with the whole class's runs in one mlflow: everyone's handed-out
runs side by side (they should agree, up to the noise between machines and seeds), and
every study with its search space, its baseline and its final in sigmas. Your runs are part
of that picture, including the ones that lost. To browse your own first:
`uv run mlflow ui --backend-store-uri sqlite:///artefacts/mlflow/mlruns.db`, from the root
of the course repository.